# SaShiMi + DiffWave — music continuation (Colab quickstart)
Milestone 1: unconditional raw-waveform generation at 22.05 kHz / ~2 s clips on a single A100 or L4 (Colab Pro).

**Runtime → Change runtime type → GPU (A100 or L4).** Do *not* pip-install torch/torchaudio; use Colab's preinstalled CUDA-matched build.

In [ ]:
# 1. Get the project. Either clone your repo, or upload this folder to Drive and mount it.
# Replace with your repo URL if you've pushed it; otherwise upload the project to Drive.
# !git clone <YOUR_REPO_URL> diffwave-sashimi
from google.colab import drive; drive.mount('/content/drive')
%cd /content/drive/MyDrive/diffwave-sashimi   # <-- adjust to where you placed the project

In [ ]:
# 2. Confirm GPU + install Python deps (NOT torch/torchaudio).
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!pip install -q -r requirements-colab.txt

In [ ]:
# 3. Inspect / clean your audio. Put your few long tracks in raw_audio/ first.
#    This reports total duration + segment count, and writes mono 22050 Hz wavs,
#    holding out the last 10% of each track for later continuation eval.
!python scripts/prepare_data.py --in_dir raw_audio --out_dir vendor/data/music --sr 22050 --holdout_frac 0.1

In [ ]:
# 4. Train (run from vendor/ — Hydra config_path and imports are anchored there).
#    Start at d_model=64; bump to 128 once it's training and memory allows.
%cd vendor
!python train.py experiment=music train.batch_size_per_gpu=8
# Optional: wandb.mode=online  model.d_model=128  train.iters_per_ckpt=5000

In [ ]:
# 5. Generate unconditional samples from the latest checkpoint.
!python generate.py experiment=music generate.ckpt_iter=max generate.n_samples=8
# WAVs land in vendor/exp/<run>/waveforms/

In [ ]:
# 6. Listen to a generated sample.
import glob, IPython.display as ipd
wavs = sorted(glob.glob('exp/**/waveforms/**/*.wav', recursive=True))
print(wavs[-3:])
ipd.Audio(wavs[-1]) if wavs else print('No samples found yet.')